In [8]:
!pip install -q groq langchain-groq crewai crewai-tools

In [23]:
import sys
from unittest.mock import MagicMock
from groq import Groq
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
client = Groq(api_key=os.environ["GROQ_API_KEY"])

# Real completion function that calls Groq
def real_groq_completion(model, messages, **kwargs):
    model_name = model.replace("groq/", "")
    response = client.chat.completions.create(
        model=model_name,
        messages=messages,
        temperature=kwargs.get("temperature", 0.1),
        max_tokens=kwargs.get("max_tokens", 4096),
    )
    # Return object shaped like litellm response
    mock_response = MagicMock()
    mock_response.choices[0].message.content = response.choices[0].message.content
    mock_response.usage.prompt_tokens = response.usage.prompt_tokens
    mock_response.usage.completion_tokens = response.usage.completion_tokens
    return mock_response

# Build fake litellm with real Groq backend
fake_litellm = MagicMock()
fake_litellm.__version__ = "1.55.0"
fake_litellm.drop_params = True
fake_litellm.completion = real_groq_completion
fake_litellm.supports_function_calling = MagicMock(return_value=True)
fake_litellm.get_supported_openai_params = MagicMock(return_value=["temperature", "max_tokens"])

sys.modules['litellm'] = fake_litellm

# Remove crewai cache and reload
crewai_modules = [k for k in sys.modules if k.startswith('crewai')]
for mod in crewai_modules:
    del sys.modules[mod]

from crewai import Agent, Task, Crew, Process
from crewai_tools import CodeInterpreterTool
import crewai.llm as cm
cm.litellm = fake_litellm
cm.LITELLM_AVAILABLE = True

code_interpreter = CodeInterpreterTool()
print("Setup complete - real Groq backend connected")

Setup complete - real Groq backend connected


In [24]:
import crewai.llm as cm
import sys
from unittest.mock import MagicMock

# Force the flag
cm.LITELLM_AVAILABLE = True

# Patch __new__ to bypass the check entirely
_original_new = cm.LLM.__new__

def _patched_new(cls, model=None, is_litellm=None, **kwargs):
    cm.LITELLM_AVAILABLE = True
    # Skip the LITELLM check by calling object.__new__ directly
    instance = object.__new__(cls)
    return instance

cm.LLM.__new__ = _patched_new

# Also patch the utility that creates LLMs from strings
import crewai.utilities.llm_utils as lu

def patched_create_llm(llm_value):
    if isinstance(llm_value, str):
        # Use fake litellm mock for groq
        mock = MagicMock()
        mock.model = llm_value
        mock.call = MagicMock()
        return mock
    return llm_value

lu.create_llm = patched_create_llm

print("Patched successfully")
print("LITELLM_AVAILABLE:", crewai.llm.LITELLM_AVAILABLE)

Patched successfully
LITELLM_AVAILABLE: True


In [25]:
import sys
from unittest.mock import MagicMock
import crewai.llm as cm
import crewai.utilities.llm_utils as lu

# Create a proper fake litellm with all required attributes
fake_litellm = MagicMock()
fake_litellm.__version__ = "1.55.0"
fake_litellm.drop_params = True
fake_litellm.completion = MagicMock()
fake_litellm.supports_function_calling = MagicMock(return_value=True)
fake_litellm.get_supported_openai_params = MagicMock(return_value=[])
fake_litellm.utils = MagicMock()

# Inject into sys.modules
sys.modules['litellm'] = fake_litellm

# Patch the litellm reference inside crewai.llm module directly
cm.litellm = fake_litellm
cm.LITELLM_AVAILABLE = True

print("litellm.drop_params:", fake_litellm.drop_params)
print("Patched successfully")

litellm.drop_params: True
Patched successfully


In [26]:
import os
from google.colab import userdata
from crewai import Agent, Task, Crew, Process
from crewai_tools import CodeInterpreterTool

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
code_interpreter = CodeInterpreterTool()

code_analyzer = Agent(
    role="Code Analyzer",
    goal="Identify all syntax and logical errors in the provided Python code.",
    backstory="You are an expert Python developer who carefully inspects code and produces a detailed list of every error found.",
    tools=[code_interpreter],
    llm="groq/llama-3.3-70b-versatile",
    verbose=True,
    allow_delegation=False
)

code_corrector = Agent(
    role="Code Corrector",
    goal="Fix all identified errors in the Python code and return a corrected version.",
    backstory="You are a skilled Python engineer who excels at debugging and correcting code.",
    llm="groq/llama-3.3-70b-versatile",
    verbose=True,
    allow_delegation=False
)

manager = Agent(
    role="Manager",
    goal="Oversee the code analysis and correction process.",
    backstory="You are a seasoned software engineering manager coordinating analysis and correction of Python code.",
    llm="groq/llama-3.3-70b-versatile",
    verbose=True,
    allow_delegation=True
)

print("Agents initialized successfully")

Agents initialized successfully


In [30]:
from groq import Groq
import os

client = Groq(api_key=os.environ["GROQ_API_KEY"])

def make_real_call(model_name):
    def real_call(messages, **kwargs):
        clean_model = model_name.replace("groq/", "")
        # Filter messages to only valid roles
        clean_messages = [
            {"role": m["role"], "content": m["content"]}
            for m in messages
            if isinstance(m, dict) and "role" in m and "content" in m
        ]
        response = client.chat.completions.create(
            model=clean_model,
            messages=clean_messages,
            temperature=kwargs.get("temperature", 0.1),
            max_tokens=kwargs.get("max_tokens", 4096),
        )
        return response.choices[0].message.content
    return real_call

# Patch the call method on each agent's LLM instance directly
for agent in [code_analyzer, code_corrector, manager]:
    agent.llm.call = make_real_call(agent.llm.model)
    print(f"Patched: {agent.role} -> {agent.llm.model}")

print("All agents patched with real Groq calls")

Patched: Code Analyzer -> groq/llama-3.3-70b-versatile
Patched: Code Corrector -> groq/llama-3.3-70b-versatile
Patched: Manager -> groq/llama-3.3-70b-versatile
All agents patched with real Groq calls


In [31]:
# Example buggy Python code
buggy_code = """
def fibonacci_iterative(n):
    if n < 0:
        return []
    elif n == 1:
        return [0]
    elif n == 2:
        return [0, 1]

    fib_sequence = [0, 1]
    for i in range(2, n):
    next_fib = fib_sequence[-1] + fib_sequence[-2]
    fib_sequence.append(next_fib)
    return fib_sequence
"""

# Define Tasks
analysis_task = Task(
    description=(
        f"Analyze the following Python code and identify all syntax errors, "
        f"indentation issues, and logical errors. Use the CodeInterpreterTool "
        f"to evaluate the code. Provide a detailed list of every error found.\n\n"
        f"Python Code:\n```python{buggy_code}```"
    ),
    expected_output=(
        "A detailed list of all errors found in the code, including:\n"
        "- Error type (syntax, indentation, logical)\n"
        "- Line number where the error occurs\n"
        "- Description of the error"
    ),
    agent=code_analyzer
)

correction_task = Task(
    description=(
        "Based on the errors identified by the Code Analyzer, fix all issues "
        "in the Python code. Return the complete corrected version of the code "
        "that runs without errors and produces the correct Fibonacci sequence."
    ),
    expected_output=(
        "The fully corrected Python code with all syntax, indentation, and logical "
        "errors fixed. The code should be properly indented and executable."
    ),
    agent=code_corrector,
    context=[analysis_task]
)

In [32]:
# Create the Crew with planning enabled
crew = Crew(
    agents=[code_analyzer, code_corrector],
    tasks=[analysis_task, correction_task],
    process=Process.sequential,
    manager_agent=manager,
    planning=True,
    verbose=True
)

In [33]:
crew = Crew(
    agents=[code_analyzer, code_corrector],
    tasks=[analysis_task, correction_task],
    process=Process.sequential,
    manager_agent=manager,
    planning=False,
    verbose=True
)

result = crew.kickoff()
print("\n" + "="*60)
print("FINAL OUTPUT")
print("="*60)
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  80ce9bb9-d493-477a-8a3e-5c63a921a198                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze the following Python code and identify all syntax errors, indentation issues, and logical        │
│  errors. Use the CodeInterpreterTool to evaluate the code. Provide a detailed list of every error found.        │
│                                                                                                                 │
│  Python Code:                                                                                                   │
│  ```python                                                                                                      │
│  def fibonacci_iterative(n):                                                                                    │
│      if n < 0:                                                                                                  │
│          return []                                                                                              │
│      elif n == 1:                                                                                               │
│          return [0]                                                                                             │
│      elif n == 2:                                                                                               │
│          return [0, 1]                                                                                          │
│                                                                                                                 │
│      fib_sequence = [0, 1]                                                                                      │
│      for i in range(2, n):                                                                                      │
│      next_fib = fib_sequence[-1] + fib_sequence[-2]                                                             │
│      fib_sequence.append(next_fib)                                                                              │
│      return fib_sequence                                                                                        │
│  ```                                                                                                            │
│  ID: f47dbcc3-2d21-4336-9b06-fca91d69f133                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Code Analyzer                                                                                           │
│                                                                                                                 │
│  Task: Analyze the following Python code and identify all syntax errors, indentation issues, and logical        │
│  errors. Use the CodeInterpreterTool to evaluate the code. Provide a detailed list of every error found.        │
│                                                                                                                 │
│  Python Code:                                                                                                   │
│  ```python                                                                                                      │
│  def fibonacci_iterative(n):                                                                                    │
│      if n < 0:                                                                                                  │
│          return []                                                                                              │
│      elif n == 1:                                                                                               │
│          return [0]                                                                                             │
│      elif n == 2:                                                                                               │
│          return [0, 1]                                                                                          │
│                                                                                                                 │
│      fib_sequence = [0, 1]                                                                                      │
│      for i in range(2, n):                                                                                      │
│      next_fib = fib_sequence[-1] + fib_sequence[-2]                                                             │
│      fib_sequence.append(next_fib)                                                                              │
│      return fib_sequence                                                                                        │
│  ```                                                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Code Analyzer                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Step-by-step analysis of the problem:                                                                        │
│  1. The provided Python code is for a function named `fibonacci_iterative` that calculates the Fibonacci        │
│  sequence up to the nth number.                                                                                 │
│  2. **Indentation issue**: The lines `next_fib = fib_sequence[-1] + fib_sequence[-2]` and                       │
│  `fib_sequence.append(next_fib)` are not indented correctly. They should be indented under the `for` loop.      │
│  3. **Logical error**: The `return fib_sequence` statement is not indented correctly. It should be outside the  │
│  `for` loop, but since it's not indented, it will be executed after the first iteration of the loop, causing    │
│  the function to return after calculating only the third Fibonacci number.                                      │
│  4. **No syntax errors** were found in the provided code, aside from the indentation issues.                    │
│                                                                                                                 │
│  # Fixed solution:                                                                                              │
│  ```python                                                                                                      │
│  def fibonacci_iterative(n):                                                                                    │
│      if n < 0:                                                                                                  │
│          return []                                                                                              │
│      elif n == 1:                                                                                               │
│          return [0]                                                                                             │
│      elif n == 2:                                                                                               │
│          return [0, 1]                                                                                          │
│                                                                                                                 │
│      fib_sequence = [0, 1]                                                                                      │
│      for i in range(2, n):                                                                                      │
│          next_fib = fib_sequence[-1] + fib_sequence[-2]                                                         │
│          fib_sequence.append(next_fib)                                                                          │
│      return fib_sequence                                                                                        │
│  ```                                                                                                            │
│                                                                                                                 │
│  # Explanation of changes:                                                                                      │
│  * **Fixed indentation**: Indented `next_fib = fib_sequence[-1] + fib_sequence[-2]` and                         │
│  `fib_sequence.append(next_fib)` under the `for` loop. 

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Analyze the following Python code and identify all syntax errors, indentation issues, and logical errors. Use  │
│  the CodeInterpreterTool to evaluate the code. Provide a detailed list of every error found.                    │
│                                                                                                                 │
│  Python Code:                                                                                                   │
│  ```python                                                                                                      │
│  def fibonacci_iterative(n):                                                                                    │
│      if n < 0:                                                                                                  │
│          return []                                                                                              │
│      elif n == 1:                                                                                               │
│          return [0]                                                                                             │
│      elif n == 2:                                                                                               │
│          return [0, 1]                                                                                          │
│                                                                                                                 │
│      fib_sequence = [0, 1]                                                                                      │
│      for i in range(2, n):                                                                                      │
│      next_fib = fib_sequence[-1] + fib_sequence[-2]                                                             │
│      fib_sequence.append(next_fib)                                                                              │
│      return fib_sequence                                                                                        │
│  ```                                                                                                            │
│  Agent:                                                                                                         │
│  Code Analyzer                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Code Corrector                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```python                                                                                                      │
│  def fibonacci_iterative(n):                                                                                    │
│      if n < 0:                                                                                                  │
│          return []                                                                                              │
│      elif n == 1:                                                                                               │
│          return [0]                                                                                             │
│      elif n == 2:                                                                                               │
│          return [0, 1]                                                                                          │
│                                                                                                                 │
│      fib_sequence = [0, 1]                                                                                      │
│      for i in range(2, n):                                                                                      │
│          next_fib = fib_sequence[-1] + fib_sequence[-2]                                                         │
│          fib_sequence.append(next_fib)                                                                          │
│      return fib_sequence                                                                                        │
│                                                                                                                 │
│  # Tests and example uses:                                                                                      │
│  print(fibonacci_iterative(5))  # Output: [0, 1, 1, 2, 3]                                                       │
│  print(fibonacci_iterative(8))  # Output: [0, 1, 1, 2, 3, 5, 8, 13]                                             │
│  ```                                                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Based on the errors identified by the Code Analyzer, fix all issues in the Python code. Return the complete    │
│  corrected version of the code that runs without errors and produces the correct Fibonacci sequence.            │
│  Agent:                                                                                                         │
│  Code Corrector                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  80ce9bb9-d493-477a-8a3e-5c63a921a198                                                                           │
│  Final Output: ```python                                                                                        │
│  def fibonacci_iterative(n):                                                                                    │
│      if n < 0:                                                                                                  │
│          return []                                                                                              │
│      elif n == 1:                                                                                               │
│          return [0]                                                                                             │
│      elif n == 2:                                                                                               │
│          return [0, 1]                                                                                          │
│                                                                                                                 │
│      fib_sequence = [0, 1]                                                                                      │
│      for i in range(2, n):                                                                                      │
│          next_fib = fib_sequence[-1] + fib_sequence[-2]                                                         │
│          fib_sequence.append(next_fib)                                                                          │
│      return fib_sequence                                                                                        │
│                                                                                                                 │
│  # Tests and example uses:                                                                                      │
│  print(fibonacci_iterative(5))  # Output: [0, 1, 1, 2, 3]                                                       │
│  print(fibonacci_iterative(8))  # Output: [0, 1, 1, 2, 3, 5, 8, 13]                                             │
│  ```                                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL OUTPUT
```python
def fibonacci_iterative(n):
    if n < 0:
        return []
    elif n == 1:
        return [0]
    elif n == 2:
        return [0, 1]

    fib_sequence = [0, 1]
    for i in range(2, n):
        next_fib = fib_sequence[-1] + fib_sequence[-2]
        fib_sequence.append(next_fib)
    return fib_sequence

# Tests and example uses:
print(fibonacci_iterative(5))  # Output: [0, 1, 1, 2, 3]
print(fibonacci_iterative(8))  # Output: [0, 1, 1, 2, 3, 5, 8, 13]
```


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯